# KMeans pada LSA (ternormalisasi)
**text › clustering** · notebook `kmeans_lsa`

**Kapan dipakai:** Resep standar clustering teks: LSA lalu normalisasi sehingga KMeans mendekati cosine.

**Jebakan umum:** K harus dipilih; cluster bisa mengikuti panjang dokumen, bukan topik.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/classification_en/news4_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/classification_en/news4_test.csv"
TEXT_COL   = "text"
LABEL_COL  = "label"            # hanya untuk EVALUASI; algoritma tidak melihatnya
LANG       = "en"
USE_STOPWORDS = True
SEED = 42
MODEL_KEY  = "kmeans_lsa"
OUT = out_dir("text", "clustering", MODEL_KEY)
set_seed(SEED)
from superdb.text import ID_STOPWORDS
STOP = (ID_STOPWORDS if LANG == "id" else "english") if USE_STOPWORDS else None
K = 4      # jumlah cluster (untuk metode yang membutuhkan)

## 2. Baca data
Label hanya dipakai untuk **mengecek** hasil (ARI/NMI); clustering tidak melihatnya.

In [ ]:
from superdb.text import clean_text
df = pd.read_csv(TRAIN_PATH); texts = df[TEXT_COL].map(clean_text).tolist(); true = df[LABEL_COL].to_numpy()
print(len(texts), "dokumen,", len(set(true)), "kategori asli")

## 3. Representasi dan clustering

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans

vec = make_pipeline(TfidfVectorizer(min_df=3, max_df=0.5, stop_words=STOP, sublinear_tf=True), TruncatedSVD(n_components=100, random_state=42), Normalizer(copy=False))
model = KMeans(n_clusters=K, n_init=10, random_state=42)
Z = vec.fit_transform(texts)
nz = np.asarray(abs(Z).sum(axis=1)).ravel() > 0                # dokumen tanpa kata tersisa (vektor nol) dibuang
if (~nz).any():
    print(f"membuang {int((~nz).sum())} dokumen kosong"); Z = Z[nz]; texts = [t for t, k in zip(texts, nz) if k]; true = true[nz]
labels = model.fit_predict(Z); Zeval = Z

## 4. Kualitas cluster
**Silhouette** (tanpa label, jarak cosine) dan **NMI/ARI/purity** (terhadap kategori asli). Cluster noise (-1) tidak dihitung pada silhouette.

In [ ]:
from sklearn.metrics import silhouette_score, normalized_mutual_info_score as nmi, adjusted_rand_score as ari
from superdb.text_extra import purity
ok = labels != -1
sil = float(silhouette_score(Zeval[ok], labels[ok], metric="cosine")) if len(set(labels[ok])) > 1 else float("nan")
metrics = {"silhouette_cosine": sil, "NMI": nmi(true, labels), "ARI": ari(true, labels), "purity": purity(true, labels), "n_cluster": len(set(labels) - {-1})}
display(show_table(metrics))
display(pd.crosstab(pd.Series(labels, name="cluster"), pd.Series(true, name="kategori asli")))

## 5. Kata khas tiap cluster
Dari rata-rata bobot TF-IDF dokumen dalam cluster. Ini bahan untuk **memberi nama cluster** di laporan.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer as TV
tv = TV(min_df=3, max_df=0.5, stop_words=STOP, sublinear_tf=True); T = tv.fit_transform(texts); tn = np.asarray(tv.get_feature_names_out())
top = {}
for c in sorted(set(labels)):
    m_ = np.asarray(T[labels == c].mean(0)).ravel(); top[f"cluster {c} (n={int((labels == c).sum())})"] = list(tn[np.argsort(-m_)[:8]])
display(pd.DataFrame(top).T)

## 6. Memilih jumlah cluster (silhouette vs K)

In [ ]:
from sklearn.base import clone
rows = []
for k in range(2, 9):
    m_ = clone(model); m_.set_params(**{("n_components" if hasattr(m_, "n_components") and not hasattr(m_, "n_clusters") else "n_clusters"): k})
    lab = m_.fit_predict(Zeval) if hasattr(m_, "fit_predict") else m_.fit(Zeval).predict(Zeval)
    rows.append({"K": k, "silhouette": silhouette_score(Zeval, lab, metric="cosine"), "NMI": nmi(true, lab)})
sweep = pd.DataFrame(rows).set_index("K"); display(sweep.round(3))
ax = sweep.plot(marker="o", figsize=(6, 3.2), title="Kualitas vs K"); plt.tight_layout(); plt.savefig(OUT / "sweep_k.png", dpi=130); plt.show()

In [ ]:
save_json({"metrics": metrics, "top_terms": top}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Silhouette tertinggi belum tentu cluster paling berguna; baca kata khas dan contoh dokumen.
- Embedding kalimat (heavy) biasanya memisahkan topik lebih rapi daripada TF-IDF.